In [15]:
import os
import re
from Bio.PDB import PDBParser, PDBIO, Select, Superimposer
from Bio.SeqUtils import seq1
from Bio import pairwise2
import numpy as np

def parse_biomolecule_chains(pdb_file):
    with open(pdb_file, 'r') as file:
        pdb_lines = file.readlines()

    biomolecules = {}
    current_biomolecule = None

    for line in pdb_lines:
        if line.startswith('REMARK 350 BIOMOLECULE:'):
            current_biomolecule = int(line.strip().split(':')[1])
            biomolecules[current_biomolecule] = []
        elif line.startswith('REMARK 350 APPLY THE FOLLOWING TO CHAINS:'):
            # Извлекаем идентификаторы цепей
            chains = re.findall(r'[A-Za-z0-9]+', line.split(':')[1])
            biomolecules[current_biomolecule].extend(chains)

    return biomolecules

def split_biomolecule_pdb(pdb_file):
    pdb_name = os.path.splitext(os.path.basename(pdb_file))[0]
    # Парсим информацию о биомолекулах и их цепях
    biomolecules = parse_biomolecule_chains(pdb_file)
    
    with open(pdb_file, 'r') as file:
        pdb_lines = file.readlines()

    # Сохраняем заголовок и концовку PDB файла
    header_lines = []
    footer_lines = []
    atom_section_started = False
    for line in pdb_lines:
        if line.startswith('ATOM') or line.startswith('HETATM'):
            atom_section_started = True
            break
        header_lines.append(line)

    for line in pdb_lines[::-1]:
        if line.startswith('ATOM') or line.startswith('HETATM'):
            break
        footer_lines.insert(0, line)

    biomolecule_files = []

    # Обрабатываем атомные строки и сохраняем цепи для каждой биомолекулы
    for biomol_id, chains in biomolecules.items():
        biomol_atom_lines = []
        atom_section_started = False
        for line in pdb_lines:
            if line.startswith('ATOM') or line.startswith('HETATM'):
                atom_section_started = True
                chain_id = line[21].strip()  # Идентификатор цепи (в 22-й колонке PDB)
                if chain_id in chains:
                    biomol_atom_lines.append(line)
            elif atom_section_started and line.startswith('TER'):
                # Добавляем 'TER' для завершения цепи
                chain_id = line[21].strip()
                if chain_id in chains:
                    biomol_atom_lines.append(line)
            elif atom_section_started and line.startswith('ENDMDL'):
                biomol_atom_lines.append(line)
                break

        if not biomol_atom_lines:
            print(f"Предупреждение: биомолекула {biomol_id} не содержит атомных записей.")
            continue

        # Сохраняем каждую биомолекулу в отдельный файл с корректным названием
        biomol_file = f'{pdb_name}_bioml_{biomol_id}.pdb'
        with open(biomol_file, 'w') as file:
            file.writelines(header_lines + biomol_atom_lines + footer_lines)
        print(f"Биомолекула {biomol_id} с цепями {chains} сохранена в {biomol_file}")
        biomolecule_files.append((biomol_id, biomol_file, chains))

    return biomolecule_files

def get_sequences(structure):
    sequences = {}
    for model in structure:
        for chain in model:
            seq = ''
            for residue in chain:
                if residue.id[0] == ' ':
                    seq += seq1(residue.resname)
            sequences[chain.id] = seq
    return sequences

def are_sequences_identical(seq_dict1, seq_dict2):
    # Сравниваем последовательности всех цепей, очищая их от пробелов и символов
    if set(seq_dict1.keys()) != set(seq_dict2.keys()):
        return False

    identical = True
    for chain_id in seq_dict1.keys():
        seq1 = seq_dict1[chain_id].replace(" ", "").strip()
        seq2 = seq_dict2[chain_id].replace(" ", "").strip()
        if seq1 != seq2:
            print(f"  Несоответствие в цепи {chain_id}:")
            print(f"  Последовательность 1: {seq1}")
            print(f"  Последовательность 2: {seq2}")
            identical = False
        else:
            print(f"  Цепь {chain_id} идентична.")
    
    return identical

def calculate_rmsd(structure1, structure2):
    # Предполагаем, что структура имеет только одну модель
    fixed_atoms = []
    moving_atoms = []
    for chain1 in structure1[0]:
        if chain1.id in structure2[0]:
            chain2 = structure2[0][chain1.id]
            for residue1, residue2 in zip(chain1.get_residues(), chain2.get_residues()):
                for atom1_name in residue1.child_dict:
                    if atom1_name in residue2.child_dict:
                        atom1 = residue1[atom1_name]
                        atom2 = residue2[atom1_name]
                        fixed_atoms.append(atom1)
                        moving_atoms.append(atom2)
    if not fixed_atoms or not moving_atoms:
        return None
    sup = Superimposer()
    sup.set_atoms(fixed_atoms, moving_atoms)
    rmsd = sup.rms
    return rmsd

def remove_duplicate_biomolecules(biomolecule_files, rmsd_threshold=0.5):
    parser = PDBParser(QUIET=True)
    unique_biomolecules = []
    duplicates = []
    for i, (biomol_id_i, biomol_file_i, chains_i) in enumerate(biomolecule_files):
        structure_i = parser.get_structure(f'biomol_{biomol_id_i}', biomol_file_i)
        seq_i = get_sequences(structure_i)
        
        # Вывод информации о цепях и последовательностях
        print(f"\nБиомолекула {biomol_id_i}:")
        for chain_id, sequence in seq_i.items():
            print(f"  Цепь {chain_id}, последовательность: {sequence}")

        is_duplicate = False
        for j, (biomol_id_j, biomol_file_j, chains_j) in enumerate(unique_biomolecules):
            structure_j = parser.get_structure(f'biomol_{biomol_id_j}', biomol_file_j)
            seq_j = get_sequences(structure_j)
            
            # Проверяем идентичность последовательностей
            print(f"\nСравнение с биомолекулой {biomol_id_j}:")
            identical_sequences = are_sequences_identical(seq_i, seq_j)
            if identical_sequences:
                print(f"  Последовательности цепей идентичны между биомолекулами {biomol_id_i} и {biomol_id_j}.")
                # Проверяем RMSD
                rmsd = calculate_rmsd(structure_i, structure_j)
                if rmsd is not None and rmsd <= rmsd_threshold:
                    print(f"  Биомолекула {biomol_id_i} является дубликатом биомолекулы {biomol_id_j} (RMSD = {rmsd:.3f}).")
                    is_duplicate = True
                    duplicates.append(biomol_file_i)
                    break
            else:
                print(f"  Последовательности цепей различаются между биомолекулами {biomol_id_i} и {biomol_id_j}.")
        
        if not is_duplicate:
            unique_biomolecules.append((biomol_id_i, biomol_file_i, chains_i))

    # Удаляем файлы дубликатов
    for duplicate_file in duplicates:
        os.remove(duplicate_file)
        print(f"\nУдален дубликат файла {duplicate_file}")

    print("\nОставшиеся уникальные биомолекулы:")
    for biomol_id, biomol_file, chains in unique_biomolecules:
        print(f"Биомолекула {biomol_id}: файл {biomol_file}, цепи {chains}")

def process_pdb_file(pdb_file):
    # Шаг 1: Разделяем PDB-файл на биомолекулы
    biomolecule_files = split_biomolecule_pdb(pdb_file)

    # Шаг 2: Удаляем дубликаты биомолекул
    remove_duplicate_biomolecules(biomolecule_files)

# Пример использования
pdb_file = '1gc4.pdb'  # Замените на путь к вашему PDB-файлу
process_pdb_file(pdb_file)


Биомолекула 1 с цепями ['A', 'B'] сохранена в 1gc4_bioml_1.pdb
Биомолекула 2 с цепями ['C', 'D'] сохранена в 1gc4_bioml_2.pdb

Биомолекула 1:
  Цепь A, последовательность: MRGLSRRVQAMKPDAVVAVNAKALELRRQGVDLVALTAGEPDFDTPEHVKEAARRALAQGKTKYAPPAGIPELREALAEKFRRENGLSVTPEETIVTVGGSQALFNLFQAILDPGDEVIVLSPYWVSYPEMVRFAGGVVVEVETLPEEGFVPDPERVRRAITPRTKALVVNSPNNPTGAVYPKEVLEALARLAVEHDFYLVSDEIYEHLLYEGEHFSPGRVAPEHTLTVNGAAKAFAMTGWRIGYACGPKEVIKAMASVSRQSTTSPDTIAQWATLEALTNQEASRAFVEMAREAYRRRRDLLLEGLTALGLKAVRPSGAFYVLMDTSPIAPDEVRAAERLLEAGVAVVPGTDFAAFGHVRLSYATSEENLRKALERFARVL
  Цепь B, последовательность: MRGLSRRVQAMKPDAVVAVNAKALELRRQGVDLVALTAGEPDFDTPEHVKEAARRALAQGKTKYAPPAGIPELREALAEKFRRENGLSVTPEETIVTVGGSQALFNLFQAILDPGDEVIVLSPYWVSYPEMVRFAGGVVVEVETLPEEGFVPDPERVRRAITPRTKALVVNSPNNPTGAVYPKEVLEALARLAVEHDFYLVSDEIYEHLLYEGEHFSPGRVAPEHTLTVNGAAKAFAMTGWRIGYACGPKEVIKAMASVSRQSTTSPDTIAQWATLEALTNQEASRAFVEMAREAYRRRRDLLLEGLTALGLKAVRPSGAFYVLMDTSPIAPDEVRAAERLLEAGVAVVPGTDFAAFGHVRLSYATSEENLRKALERFARVL

Биомолекула 2:
  Цепь C, послед

In [20]:
import os
import re
from Bio.PDB import PDBParser, Superimposer
from Bio.Seq import Seq
from Bio.SeqUtils import seq1
from Bio import pairwise2
import numpy as np

def parse_biomolecule_chains(pdb_file):
    with open(pdb_file, 'r') as file:
        pdb_lines = file.readlines()

    biomolecules = {}
    current_biomolecule = None

    for line in pdb_lines:
        if line.startswith('REMARK 350 BIOMOLECULE:'):
            current_biomolecule = int(line.strip().split(':')[1])
            biomolecules[current_biomolecule] = []
        elif line.startswith('REMARK 350 APPLY THE FOLLOWING TO CHAINS:'):
            chains = re.findall(r'[A-Za-z0-9]+', line.split(':')[1])
            biomolecules[current_biomolecule].extend(chains)

    return biomolecules

def split_biomolecule_pdb(pdb_file):
    # Получаем имя файла без расширения
    pdb_name = os.path.splitext(os.path.basename(pdb_file))[0]

    # Парсим информацию о биомолекулах и их цепях
    biomolecules = parse_biomolecule_chains(pdb_file)
    
    with open(pdb_file, 'r') as file:
        pdb_lines = file.readlines()

    # Сохраняем заголовок PDB файла
    header_lines = []
    footer_lines = []
    atom_section_started = False
    for line in pdb_lines:
        if line.startswith('ATOM') or line.startswith('HETATM'):
            atom_section_started = True
            break
        header_lines.append(line)

    for line in pdb_lines[::-1]:
        if line.startswith('ATOM') or line.startswith('HETATM'):
            break
        footer_lines.insert(0, line)

    # Обрабатываем атомные строки и сохраняем цепи для каждой биомолекулы
    biomolecule_files = []
    for biomol_id, chains in biomolecules.items():
        biomol_atom_lines = []
        atom_section_started = False
        for line in pdb_lines:
            if line.startswith('ATOM') or line.startswith('HETATM'):
                atom_section_started = True
                chain_id = line[21]  # Идентификатор цепи (в 22-й колонке PDB)
                if chain_id in chains:
                    biomol_atom_lines.append(line)
            elif atom_section_started and line.startswith('TER'):
                # Добавляем 'TER' для завершения цепи
                chain_id = line[21]
                if chain_id in chains:
                    biomol_atom_lines.append(line)
            elif atom_section_started and line.startswith('ENDMDL'):
                biomol_atom_lines.append(line)
                break

        # Сохраняем каждую биомолекулу в отдельный файл
        biomol_file = f'{pdb_name}_bioml_{biomol_id}.pdb'
        with open(biomol_file, 'w') as file:
            file.writelines(header_lines + biomol_atom_lines + footer_lines)
        biomolecule_files.append(biomol_file)
        print(f"Биомолекула {biomol_id} с цепями {chains} сохранена в {biomol_file}")

    return biomolecule_files

def get_chain_sequences(structure):
    sequences = {}
    for model in structure:
        for chain in model:
            seq = ''
            for residue in chain:
                if residue.id[0] == ' ':
                    seq += seq1(residue.resname)
            sequences[chain.id] = Seq(seq)
            print(f"Последовательность цепи {chain.id}: {sequences[chain.id]}")
    return sequences

def are_structures_identical(structure1, structure2):
    # Получаем цепи из обеих структур
    chains1 = list(structure1.get_chains())
    chains2 = list(structure2.get_chains())

    if len(chains1) != len(chains2):
        print("Количество цепей не совпадает.")
        return False

    # Сравниваем последовательности
    sequences1 = get_chain_sequences(structure1)
    sequences2 = get_chain_sequences(structure2)

    for chain_id1, seq1 in sequences1.items():
        found = False
        for chain_id2, seq2 in sequences2.items():
            alignments = pairwise2.align.globalxx(seq1, seq2)
            alignment = alignments[0]
            identity = alignment[2] / alignment[4] * 100
            if identity == 100.0:
                # Проверяем RMSD
                ca_atoms1 = [atom for atom in structure1[0][chain_id1].get_atoms() if atom.get_id() == 'CA']
                ca_atoms2 = [atom for atom in structure2[0][chain_id2].get_atoms() if atom.get_id() == 'CA']

                if len(ca_atoms1) != len(ca_atoms2):
                    continue

                sup = Superimposer()
                sup.set_atoms(ca_atoms1, ca_atoms2)
                rms = sup.rms
                print(f"RMSD между цепями {chain_id1} и {chain_id2}: {rms:.3f} Å")
                if rms < 0.5:  # Порог для определения идентичности структур
                    found = True
                    break
        if not found:
            return False
    return True

def remove_duplicate_biomolecules(biomolecule_files):
    unique_files = []
    duplicates = []

    for i, file1 in enumerate(biomolecule_files):
        parser = PDBParser(QUIET=True)
        structure1 = parser.get_structure(f'biomol_{i+1}', file1)
        is_duplicate = False
        for j, file2 in enumerate(unique_files):
            structure2 = parser.get_structure(f'unique_{j+1}', file2)
            print(f"Сравниваем {file1} и {file2}")
            if are_structures_identical(structure1, structure2):
                print(f"{file1} является дубликатом {file2}")
                duplicates.append(file1)
                is_duplicate = True
                break
        if not is_duplicate:
            unique_files.append(file1)
    
    for dup_file in duplicates:
        os.remove(dup_file)
        print(f"Удален дубликат: {dup_file}")

    print("Остались следующие уникальные биомолекулы:")
    for file in unique_files:
        print(file)

def process_pdb_file(pdb_file):
    biomolecule_files = split_biomolecule_pdb(pdb_file)
    print("\nРазделение PDB-файла на биомолекулы завершено.\n")

    remove_duplicate_biomolecules(biomolecule_files)
    print("\nУдаление дубликатов завершено.\n")



In [ ]:
process_pdb_file('1gc4.pdb')

In [21]:
process_pdb_file('1xxa.pdb')

Биомолекула 1 с цепями ['A', 'B', 'C', 'D', 'E', 'F'] сохранена в 1xxa_bioml_1.pdb

Разделение PDB-файла на биомолекулы завершено.

Остались следующие уникальные биомолекулы:
1xxa_bioml_1.pdb

Удаление дубликатов завершено.



In [22]:
process_pdb_file('5ivq.pdb')

Биомолекула 1 с цепями ['A', 'B'] сохранена в 5ivq_bioml_1.pdb

Разделение PDB-файла на биомолекулы завершено.

Остались следующие уникальные биомолекулы:
5ivq_bioml_1.pdb

Удаление дубликатов завершено.

